# 2. Модели прогноза
По первой половине записей всех датчиков предсказываем вторую. Прогноз $\hat Y$ строим по записям с шумом датчиков и сравниваем с записью $Y$ без шума. Ошибка на второй половине
$$\varepsilon=\frac{\lVert\hat Y-Y\rVert}{\lVert Y\rVert},$$
у нулевого прогноза $\varepsilon=1$.
- **M1.** Волновое уравнение: подбираем источники и контраст среды.
- **M2.** Уравнение Гельмгольца: на каждой частоте раскладываем поле по решениям этого уравнения.
- **M3.** Только записи: сумма синусоид (Фурье) или нейросеть (автоэнкодер).
- **M4.** Только записи: линейная рекуррента (SSA, MSSA).

In [1]:
import sys, warnings
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
warnings.filterwarnings("ignore")
sys.path.insert(0, "../synthetic")
from scenarios import SCENARIOS, simulate
from models import MODELS
pio.renderers.default = "plotly_mimetype+notebook_connected"

### Частоты и амплитуды (M1, M2)
M1 и M2 прогнозируют суммой синусоид, как M3 Фурье; уравнение только связывает амплитуды разных датчиков.
- Частоты $\nu_j$: пики спектра, уточнённые методом наименьших квадратов (МНК).
- Амплитуды $Z_{ij}$ находим МНК по записи $y_i(t)$ датчика $i$: $y_i(t)\approx y_i^0+\operatorname{Re}\sum_jZ_{ij}e^{2\pi\mathrm i\nu_jt}$, где $y_i^0$ постоянная составляющая. Столбец $Z_j$ содержит амплитуды всех датчиков на частоте $\nu_j$.
- Модель заменяет $Z_j$ своими $\hat Z_j$ и продолжает сумму на вторую половину.

### M1. Волновое уравнение
**Идея.** Известны уравнение и тип среды: однородная, слоистая, линза или волновод. Положения источников и контраст среды подбираем так, чтобы решение уравнения совпало с записями.

**Модель.** Источники $f_j(t)=\operatorname{Re}a_je^{2\pi\mathrm i\nu_jt}$ в точках $r_j$, $j=1,2$, возбуждают поле $u$ в среде с показателем преломления $n(r)$ и контрастом $\alpha$:
$$u_{tt}=\frac{1}{n(r)^2}\,\Delta u+\sum_{j=1}^{2}f_j(t)\,\delta(r-r_j).$$
Неизвестны положения $r_1,r_2$, контраст $\alpha$ и амплитуды $a_1,a_2$.

**Что сравниваем.** Численно решаем уравнение с источником единичной амплитуды в точке $r_j$ и получаем амплитуды датчиков $G_j(r_j,\alpha)$ на частоте $\nu_j$. Ищем
$$\min_{r_1,r_2,\alpha,a_1,a_2}\ \sum_{j=1}^{2}\bigl\lVert a_jG_j(r_j,\alpha)-Z_j\bigr\rVert^2,$$
прогноз строим по $\hat Z_j=a_jG_j$.

**Как решаем.**
1. При заданных $r_j,\alpha$ лучшую амплитуду находим явно, выделив полный квадрат:
$$\lVert aG-Z\rVert^2=\lVert G\rVert^2\Bigl|a-\frac{G^{*}Z}{\lVert G\rVert^2}\Bigr|^2+\lVert Z\rVert^2-\frac{|G^{*}Z|^2}{\lVert G\rVert^2},$$
   минимум при $a_j=G_j^{*}Z_j/\lVert G_j\rVert^2$, то есть $a_jG_j$ совпадает с проекцией $Z_j$ на направление $G_j$.
2. Начальное приближение: лучший узел грубой сетки по $r_j$ и $\alpha$.
3. Уточняем $p=(r_1,r_2,\alpha)$ методом Левенберга и Марквардта. Вещественные и мнимые части невязок $a_jG_j-Z_j$ составляют вектор $F(p)$ с матрицей Якоби $\mathcal J$. Итерация $s$ с параметром $\lambda>0$:
$$p_{s+1}=p_s-\bigl(\mathcal J^{\mathsf T}\mathcal J+\lambda I\bigr)^{-1}\mathcal J^{\mathsf T}F(p_s).$$

### M2. Уравнение Гельмгольца
Подставим колебание на одной частоте $u=\operatorname{Re}U(r)\,e^{2\pi\mathrm i\nu t}$ в волновое уравнение вне источников $u_{tt}=c^2\Delta u$ со скоростью $c$. Получим уравнение Гельмгольца для амплитуды $U$:
$$\Delta U+k^2U=0,\qquad k=\frac{2\pi\nu}{c}.$$
В неоднородной среде $c$ меняется, и уравнение выполняется лишь приближённо. Амплитуду $U$ раскладываем по решениям; начало координат в центре сетки датчиков, $(\rho,\varphi)$ полярные координаты:
- 1D: $\cos kx$, $\sin kx$;
- 2D: $J_m(k\rho)\,e^{\mathrm im\varphi}$ с функциями Бесселя $J_m$, $|m|\le3$;
- 3D: произведения сферических функций Бесселя и сферических гармоник, 9 членов.

Коэффициенты разложения находим МНК по амплитудам датчиков $Z_j$, скорость $c$ подбираем перебором. Значения разложения в точках датчиков дают $\hat Z_j$.

### M3. Фурье
Запись каждого датчика приближаем отдельно суммой её сильнейших колебаний, не больше 8. Частоты и коэффициенты находим МНК:
$$y_i(t)\approx y_i^0+\sum_j\bigl(A_{ij}\cos2\pi\nu_{ij}t+B_{ij}\sin2\pi\nu_{ij}t\bigr).$$

### M3. Автоэнкодер
Показания всех $N$ датчиков $\mathbf y_t\in\mathbb R^N$ кодируем вектором $z_t\in\mathbb R^8$, который за шаг по времени меняется линейно:
$$z_t=E(\mathbf y_t),\qquad z_{t+1}\approx Kz_t,\qquad \mathbf y_t\approx D(z_t).$$
Кодировщик $E$ и декодировщик $D$ имеют по одному скрытому слою из 64 нейронов:
$$E(\mathbf y)=W_2\tanh(W_1\mathbf y+\beta_1)+\beta_2,\qquad D(z)=V_2\tanh(V_1z+\beta_3)+\beta_4.$$
$E$, $D$, $K$ обучаются вместе: минимизируется сумма средних квадратов невязок в $\mathbf y_t\approx D(z_t)$, $z_{t+1}\approx Kz_t$ и $\mathbf y_{t+1}\approx D(Kz_t)$. Если спектральный радиус $K$ после обучения больше 1, $K$ делим на него, чтобы прогноз не рос без предела.

Прогноз на $h$ шагов после последнего отсчёта $T$ первой половины: $\hat{\mathbf y}_{T+h}=D(K^hz_T)$.

### M4. SSA и MSSA
Сумма синусоид удовлетворяет линейной рекурренте: каждый отсчёт равен одной и той же линейной комбинации фиксированного числа предыдущих. SSA находит эту рекурренту по первой половине записи одного датчика, ряду $y(1),\dots,y(T)$:
1. Все отрезки ряда длины $L=\lfloor T/2\rfloor$ (окно) ставим столбцами траекторной матрицы:
$$\begin{pmatrix} y(1)&y(2)&\cdots&y(T-L+1)\\ y(2)&y(3)&\cdots&y(T-L+2)\\ \vdots&\vdots&&\vdots\\ y(L)&y(L+1)&\cdots&y(T)\end{pmatrix}.$$
   При таком окне строк и столбцов примерно поровну.
2. Её главные левые сингулярные векторы $v_1,\dots,v_R$, взятые до наибольшего скачка сингулярных чисел, задают подпространство сигнала.
3. Траекторную матрицу проецируем на это подпространство и усредняем по антидиагоналям: получаем сглаженный ряд.
4. Отделим у $v_q$ последнюю координату $\gamma_q$, остальные обозначим $\underline v_q$. Сглаженный ряд продолжаем рекуррентой
$$y(t)=\sum_{l=1}^{L-1}w_l\,y(t-L+l),\qquad w=\frac{\sum_q\gamma_q\underline v_q}{1-\sum_q\gamma_q^2}.$$

MSSA делает то же для всех датчиков сразу: их траекторные матрицы ставим рядом, подпространство и веса $w$ общие.

## Расчёт

In [2]:
rng = np.random.default_rng(0)
err = np.full((len(SCENARIOS), len(MODELS)), np.nan)
for i, sc in enumerate(SCENARIOS.values()):
    run = simulate(sc)
    Y, t, n = run["Y"], run["t"], len(run["t"]) // 2
    noisy = Y + 0.02 * np.abs(Y).max() * rng.standard_normal(Y.shape)
    for j, forecast in enumerate(MODELS.values()):
        pred = forecast(t[:n], noisy[:, :n], run["sensors"], t[n:], sc)
        if pred is not None:
            err[i, j] = np.linalg.norm(pred - Y[:, n:]) / np.linalg.norm(Y[:, n:])

In [3]:
models = list(MODELS)
text = [["нет" if np.isnan(v) else f"{v:.3f}" for v in row] for row in err]
ticks = [0.003, 0.01, 0.03, 0.1, 0.3, 1]
fig = go.Figure(go.Heatmap(
    z=np.log10(np.clip(err, 2e-3, 1)), x=models, y=list(SCENARIOS), text=text, texttemplate="%{text}", textfont_size=12,
    colorscale="YlOrRd", zmin=np.log10(2e-3), zmax=0, xgap=1, ygap=1,
    colorbar=dict(title=dict(text="ε, лог.", font_size=12), tickvals=np.log10(ticks), ticktext=[str(v) for v in ticks], len=0.9),
    hovertemplate="%{y}<br>%{x}<br>ε = %{text}<extra></extra>"))
for y in (4.5, 8.5):
    fig.add_hline(y=y, line_color="white", line_width=4)
fig.update_layout(template="plotly_white", height=500, margin=dict(l=10, r=10, t=80, b=10), font_size=13,
                  title="Ошибка прогноза ε (у нулевого прогноза ε = 1)",
                  xaxis=dict(side="top", showgrid=False, tickvals=models, ticktext=[m.replace(" ", "<br>") for m in models]),
                  yaxis=dict(autorange="reversed", showgrid=False))
fig.show()